# Corte 1: bronze y desacoplamiento de almacenamiento y cómputo

Por: Gabriel Rodriguez y Jonatan Palomares

Parte del notebook original `Proyecto_PAMD_SECOP` (Punto 1).

## Configuración

Las rutas están en `src/pamd/config.py` y la sesión de Spark con MinIO en `src/pamd/spark_session.py`. Las credenciales vienen de las variables de entorno definidas en `.env`.

In [ ]:
# Importación de librerías

import os
from time import perf_counter
from statistics import median
import random
import math
import json
import urllib.request
from IPython.display import display, Markdown

import matplotlib.pyplot as plt

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.types import DateType, TimestampType
from pyspark import StorageLevel

from pamd.config import (
    INPUT_CSV,
    BRONZE_PARQUET_UNPARTITIONED,
    BRONZE_PARQUET,
    QUARANTINE_PARQUET,
    WRITE_MODE,
    OUTPUT_PARTITIONS,
)
from pamd.spark_session import get_spark

# Spark conectado a MinIO; las credenciales se leen del archivo .env
spark = get_spark()


In [7]:
memoria_total_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3

print("Versión de Spark:", spark.version)
print(f"Memoria disponible en Docker: {memoria_total_gb:.1f} GB")

Versión de Spark: 3.5.1
Memoria disponible en Docker: 7.6 GB


## Punto 1: Desacoplamiento de Almacenamiento y Cómputo

**Objetivo.** Observar cómo el formato y la organización física de los datos afectan las consultas cuando el almacenamiento está separado del motor de procesamiento. MinIO conserva los archivos y Spark realiza la lectura, las transformaciones y las agregaciones mediante el conector S3A.

Se parte del CSV de SECOP II, se normalizan sus datos y se utiliza una capa bronze en Parquet. Se comparan tres alternativas: CSV con limpieza durante la consulta, Parquet sin particionar y Parquet particionado por año de publicación y departamento. Además de los tiempos de consulta, se examinan el espacio ocupado y el costo de escritura.

### Lectura de SECOP y limpieza

Antes de comparar las consultas se define un tratamiento común para los datos: nombres de columnas consistentes, valores faltantes, tipos y reglas de calidad.

#### Lectura y normalización de nombres de columnas

El CSV se lee con un esquema explícito de texto para controlar posteriormente las conversiones, sin depender de la inferencia automática. El modo permisivo y la columna `_corrupt_record` permiten identificar registros malformados según las opciones de lectura. Se asume un registro por línea (`multiLine=False`).

El diccionario `raw_to_clean` traduce los encabezados originales a nombres uniformes, utilizados en el resto del notebook. `printSchema()` permite revisar el esquema antes de continuar.

In [9]:
# Diccionario con mapeo de nombre_columna_original:nombre_columna_normalizado

raw_to_clean = {
    "Entidad": "entidad",
    "Nit Entidad": "nit_entidad",
    "Departamento Entidad": "departamento_entidad",
    "Ciudad Entidad": "ciudad_entidad",
    "OrdenEntidad": "orden_entidad",
    "Entidad Centralizada": "entidad_centralizada",
    "ID del Proceso": "id_del_proceso",
    "Referencia del Proceso": "referencia_del_proceso",
    "PCI": "codigo_pci",
    "ID del Portafolio": "id_del_portafolio",
    "Nombre del Procedimiento": "nombre_del_procedimiento",
    "Descripción del Procedimiento": "descripcion_del_procedimiento",
    "Fase": "fase",
    "Fecha de Publicacion del Proceso": "fecha_publicacion_proceso",
    "Fecha de Ultima Publicación": "fecha_ultima_publicacion",
    "Fecha de Publicacion (Fase Planeacion Precalificacion)": "fecha_fase_planeacion_precalificacion",
    "Fecha de Publicacion (Fase Seleccion Precalificacion)": "fecha_fase_seleccion_precalificacion",
    "Fecha de Publicacion (Manifestacion de Interes)": "fecha_manifestacion_interes",
    "Fecha de Publicacion (Fase Borrador)": "fecha_fase_borrador",
    "Fecha de Publicacion (Fase Seleccion)": "fecha_fase_seleccion",
    "Precio Base": "precio_base",
    "Modalidad de Contratacion": "modalidad_contratacion",
    "Justificación Modalidad de Contratación": "justificacion_modalidad_contratacion",
    "Duracion": "duracion",
    "Unidad de Duracion": "unidad_duracion",
    "Fecha de Recepcion de Respuestas": "fecha_recepcion_respuestas",
    "Fecha de Apertura de Respuesta": "fecha_apertura_respuesta",
    "Fecha de Apertura Efectiva": "fecha_apertura_efectiva",
    "Ciudad de la Unidad de Contratación": "ciudad_unidad_contratacion",
    "Nombre de la Unidad de Contratación": "nombre_unidad_contratacion",
    "Proveedores Invitados": "proveedores_invitados",
    "Proveedores con Invitacion Directa": "proveedores_invitacion_directa",
    "Visualizaciones del Procedimiento": "visualizaciones_procedimiento",
    "Proveedores que Manifestaron Interes": "proveedores_manifestaron_interes",
    "Respuestas al Procedimiento": "respuestas_procedimiento",
    "Respuestas Externas": "respuestas_externas",
    "Conteo de Respuestas a Ofertas": "conteo_respuestas_ofertas",
    "Proveedores Unicos con Respuestas": "proveedores_unicos_respuestas",
    "Numero de Lotes": "numero_lotes",
    "Estado del Procedimiento": "estado_procedimiento",
    "ID Estado del Procedimiento": "id_estado_procedimiento",
    "Adjudicado": "adjudicado",
    "ID Adjudicacion": "id_adjudicacion",
    "CodigoProveedor": "codigo_proveedor",
    "Departamento Proveedor": "departamento_proveedor",
    "Ciudad Proveedor": "ciudad_proveedor",
    "Fecha Adjudicacion": "fecha_adjudicacion",
    "Valor Total Adjudicacion": "valor_total_adjudicacion",
    "Nombre del Adjudicador": "nombre_adjudicador",
    "Nombre del Proveedor Adjudicado": "nombre_proveedor_adjudicado",
    "NIT del Proveedor Adjudicado": "nit_proveedor_adjudicado",
    "Codigo Principal de Categoria": "codigo_principal_categoria",
    "Estado de Apertura del Proceso": "estado_apertura_proceso",
    "Tipo de Contrato": "tipo_contrato",
    "Subtipo de Contrato": "subtipo_contrato",
    "Categorias Adicionales": "categorias_adicionales",
    "URLProceso": "url_proceso",
    "Codigo Entidad": "codigo_entidad",
}

In [10]:
source_columns = list(raw_to_clean)

# Definición del esquema y creación de '_corrupt_record'
schema = T.StructType(
    [T.StructField(name, T.StringType(), True) for name in source_columns]
    + [T.StructField("_corrupt_record", T.StringType(), True)]
)

# Lectura de SECOP II
raw = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .option("multiLine", False)
    .option("quote", '"')
    .option("escape", '"')
    .option("encoding", "UTF-8")
    .schema(schema)
    .csv(INPUT_CSV)
)

In [11]:
# Observamos que el esquema sea el esperado

raw.printSchema()

root
 |-- Entidad: string (nullable = true)
 |-- Nit Entidad: string (nullable = true)
 |-- Departamento Entidad: string (nullable = true)
 |-- Ciudad Entidad: string (nullable = true)
 |-- OrdenEntidad: string (nullable = true)
 |-- Entidad Centralizada: string (nullable = true)
 |-- ID del Proceso: string (nullable = true)
 |-- Referencia del Proceso: string (nullable = true)
 |-- PCI: string (nullable = true)
 |-- ID del Portafolio: string (nullable = true)
 |-- Nombre del Procedimiento: string (nullable = true)
 |-- Descripción del Procedimiento: string (nullable = true)
 |-- Fase: string (nullable = true)
 |-- Fecha de Publicacion del Proceso: string (nullable = true)
 |-- Fecha de Ultima Publicación: string (nullable = true)
 |-- Fecha de Publicacion (Fase Planeacion Precalificacion): string (nullable = true)
 |-- Fecha de Publicacion (Fase Seleccion Precalificacion): string (nullable = true)
 |-- Fecha de Publicacion (Manifestacion de Interes): string (nullable = true)
 |-- Fech

In [12]:
# Se cambian los nombres de las columnas por los nombres normalizados

raw = raw.select(
    *[F.col(raw_name).alias(clean_name) for raw_name, clean_name in raw_to_clean.items()],
    "_corrupt_record",
)

#### Limpieza de datos faltantes y normalización de tipos

Se recortan espacios y se convierten las expresiones de ausencia de información a `null`. Después se transforman fechas, números y el indicador de adjudicación. Se conservan copias temporales de los valores originales para distinguir un dato ausente de una conversión fallida.

La conversión monetaria implementada interpreta los puntos como separadores de miles y acepta valores enteros antes de convertirlos a decimal; esa regla depende del formato del archivo de origen. Las reglas de calidad marcan registros con errores de sintaxis, identificador de proceso ausente, conversiones fallidas o valores numéricos negativos.

In [13]:
# Se definen los valores que representan falta de información para columnas de texto
missing_tokens = ["", "no definido", "no definida", "no adjudicado", "n/a", "na", "null"]
text_columns = list(raw_to_clean.values())

# Se crea una copia del conjunto de datos para limpiar las columnas de texto y dejar los datos originales intactos
normalized = raw
for name in text_columns:
    cleaned = F.trim(F.regexp_replace(F.col(name), r"\s+", " "))
    normalized = normalized.withColumn(
        name,
        F.when(F.lower(cleaned).isin(missing_tokens), F.lit(None)).otherwise(cleaned),
    )

# Conservar copias temporales para distinguir null legítimo de conversión fallida.
date_columns = [
    "fecha_publicacion_proceso",
    "fecha_ultima_publicacion",
    "fecha_fase_planeacion_precalificacion",
    "fecha_fase_seleccion_precalificacion",
    "fecha_manifestacion_interes",
    "fecha_fase_borrador",
    "fecha_fase_seleccion",
    "fecha_recepcion_respuestas",
    "fecha_apertura_respuesta",
    "fecha_apertura_efectiva",
    "fecha_adjudicacion",
]
decimal_columns = ["precio_base", "valor_total_adjudicacion"]
integer_columns = [
    "duracion", "proveedores_invitados", "proveedores_invitacion_directa",
    "visualizaciones_procedimiento", "proveedores_manifestaron_interes",
    "respuestas_procedimiento", "respuestas_externas",
    "conteo_respuestas_ofertas", "proveedores_unicos_respuestas",
    "numero_lotes", "codigo_pci", "nit_entidad", "id_estado_procedimiento",
    "codigo_proveedor",
]

for name in date_columns + decimal_columns + integer_columns + ["adjudicado"]:
    normalized = normalized.withColumn(f"_raw_{name}", F.col(name))

In [14]:
# Normalizamos el tipo de las columnas de fechas
for name in date_columns:
    normalized = normalized.withColumn(
        name,
        F.coalesce(
            F.to_timestamp(F.col(name), "M/d/yyyy"),
            F.to_timestamp(F.col(name), "yyyy-MM-dd'T'HH:mm:ss.SSS"),
            F.to_timestamp(F.col(name), "yyyy-MM-dd HH:mm:ss"),
            F.to_timestamp(F.col(name), "yyyy-MM-dd"),
        ),
    )

# Se normalizan las columnas con valores decimales
for name in decimal_columns:
    cleaned = F.trim(F.col(name))

    # Elimina "$" inicial y espacios posteriores
    cleaned = F.regexp_replace(cleaned, r"^\$\s*", "")

    # Elimina los puntos usados como separadores de miles
    cleaned = F.regexp_replace(cleaned, r"\.", "")

    normalized = normalized.withColumn(
        name,
        F.when(
            cleaned.rlike(r"^[+-]?\d+$"),
            cleaned.cast(T.DecimalType(20, 2)),
        ),
    )

# Se normalizan las columnas con valores enteros
for name in integer_columns:
    valid_integer = F.col(name).rlike(r"^[+-]?\d+$")
    normalized = normalized.withColumn(
        name,
        F.when(valid_integer, F.col(name).cast(T.LongType())),
    )

# Se normaliza la variable dicotómica 'adjudicado'
normalized = normalized.withColumn(
    "adjudicado",
    F.when(F.lower(F.col("adjudicado")).isin("si", "sí", "s", "true", "1"), True)
     .when(F.lower(F.col("adjudicado")).isin("no", "n", "false", "0"), False)
     .otherwise(F.lit(None).cast("boolean")),
)

In [15]:
# Razones por las que un registro es inválido
#   En caso de que '_corrupt_record' no sea nulo (anomalías de sintaxis que rompen las reglas del csv)
#   Cuando el id es nulo
reasons = [
    F.when(F.col("_corrupt_record").isNotNull(), F.lit("csv_malformado")),
    F.when(F.col("id_del_proceso").isNull(), F.lit("id_del_proceso_ausente")),
]

#   Cuando el registro de fecha original no es nulo, pero el procesado es nulo
for name in date_columns:
    reasons.append(
        F.when(
            F.col(f"_raw_{name}").isNotNull() & F.col(name).isNull(),
            F.lit(f"{name}_invalida"),
        )
    )

#   Cuando los registros numéricos originales no son nulos, pero los procesados sí lo son
for name in decimal_columns + integer_columns:
    reasons.extend([
        F.when(
            F.col(f"_raw_{name}").isNotNull() & F.col(name).isNull(),
            F.lit(f"{name}_invalido"),
        ),
        F.when(F.col(name) < 0, F.lit(f"{name}_negativo")),
    ])

#   Cuando 'adjudicado' original no es nulo, pero el procesado sí lo es
reasons.append(
    F.when(
        F.col("_raw_adjudicado").isNotNull() & F.col("adjudicado").isNull(),
        F.lit("adjudicado_invalido"),
    )
)

#   Se crea una copia de normalized para incluir los motivos de rechazo de los registros
with_quality = normalized.withColumn(
    "motivos_rechazo",
    F.filter(F.array(*reasons), lambda reason: reason.isNotNull()),
)

### Creación de bronze

Se separan los registros válidos y rechazados mediante `motivos_rechazo`. A los válidos se añaden el año de publicación y la fecha de ingesta, y se retiran las columnas auxiliares. La variante `bronze_` reemplaza las fechas anteriores a 1900 por nulos antes de escribir Parquet; el año de publicación ya fue calculado en `bronze`.

Se contemplan dos organizaciones: Parquet sin particionar y Parquet distribuido en carpetas por `anio_publicacion` y `departamento_entidad`, ambas con compresión Snappy. Filtrar por esas columnas permite a Spark descartar carpetas que no necesita leer.

Las celdas de escritura inicial y el conteo de control están comentados porque los archivos ya existen en MinIO. En una instalación nueva deben ejecutarse primero esas escrituras para crear las rutas que utilizan los siguientes puntos.

In [16]:
# Separación de los registros rechazados y válidos
rejected = with_quality.filter(F.size("motivos_rechazo") > 0)
valid = with_quality.filter(F.size("motivos_rechazo") == 0)

# Añadir columnas derivadas para el particionamiento. No se rechaza una fecha
# ausente: esos registros quedan en la partición __HIVE_DEFAULT_PARTITION__.
valid = (
    valid
    .withColumn("anio_publicacion", F.year("fecha_publicacion_proceso"))
    .withColumn("fecha_ingesta_utc", F.current_timestamp())
)

# En 'valid', se eliminan las columnas adicionales usadas en el procesamiento
temporary_columns = [
    "_corrupt_record", "motivos_rechazo",
    *[f"_raw_{name}" for name in date_columns + decimal_columns + integer_columns + ["adjudicado"]],
]
bronze = valid.drop(*temporary_columns)

In [18]:
# Detectar automáticamente columnas temporales en bronze
temporal_fields = [
    field
    for field in bronze.schema.fields
    if isinstance(field.dataType, (DateType, TimestampType))
]

print("Columnas temporales detectadas:")
for field in temporal_fields:
    print(f"- {field.name}: {field.dataType.simpleString()}")

# Reemplazar por null las fechas anteriores a 1900
# Se conservan todas las demás columnas sin modificaciones
bronze_ = bronze.select(
    *[
        (
            F.when(
                F.col(field.name) >= F.lit("1900-01-01").cast(field.dataType),
                F.col(field.name)
            )
            .otherwise(F.lit(None).cast(field.dataType))
            .alias(field.name)
        )
        if isinstance(field.dataType, (DateType, TimestampType))
        else F.col(field.name)
        for field in bronze.schema.fields
    ]
)

Columnas temporales detectadas:
- fecha_publicacion_proceso: timestamp
- fecha_ultima_publicacion: timestamp
- fecha_fase_planeacion_precalificacion: timestamp
- fecha_fase_seleccion_precalificacion: timestamp
- fecha_manifestacion_interes: timestamp
- fecha_fase_borrador: timestamp
- fecha_fase_seleccion: timestamp
- fecha_recepcion_respuestas: timestamp
- fecha_apertura_respuesta: timestamp
- fecha_apertura_efectiva: timestamp
- fecha_adjudicacion: timestamp
- fecha_ingesta_utc: timestamp


In [23]:
# Resumen de registros originales, rechazados y válidos

# raw_count = raw.count()
# valid_count = bronze.count()
# rejected_count = rejected.count()

# assert raw_count == valid_count + rejected_count, "Se perdieron filas durante el filtrado"

# print(f"Filas leídas: {raw_count}")
# print(f"Filas válidas: {valid_count}")
# print(f"Filas rechazadas: {rejected_count}")


Filas válidas: 9040863


In [27]:
# Se escribe el archivo parquet con tipos y fechas normalizados sin particionar
# Correr solo una vez

# (
#     bronze_
#     .write
#     .mode(WRITE_MODE)
#     .option("compression", "snappy")
#     .parquet(BRONZE_PARQUET_UNPARTITIONED)
# )

In [14]:
# Particionamiento por año de publicación y departamento
# Correr solo una vez

# bronze_unpartitioned = spark.read.parquet(
#    BRONZE_PARQUET_UNPARTITIONED
#)

#(
#    bronze_unpartitioned
#    .repartition(
#        OUTPUT_PARTITIONS,
#        "anio_publicacion",
#        "departamento_entidad"
#    )
#    .write
#    .mode("overwrite")
#    .option("compression", "snappy")
#    .partitionBy(
#        "anio_publicacion",
#        "departamento_entidad"
#    )
#    .parquet(BRONZE_PARQUET)
#)

### Comparación de tiempos de consulta

Se aplica la misma consulta lógica a las tres alternativas: seleccionar procesos de **Antioquia publicados en 2022**, agrupar por modalidad de contratación y calcular el número de procesos y la suma de su precio base.

El cronómetro rodea `consulta(df).collect()`: esta acción obliga a ejecutar la lectura y el cálculo y devuelve el resultado agregado al controlador. La mediana resume las repeticiones y es menos sensible a una ejecución atípica que el promedio. Se conservan tres repeticiones para CSV y veinte para cada Parquet; la tabla muestra también esa diferencia.

**Alcance de la comparación.** `csv_df = bronze` conserva el plan que lee y limpia el CSV, de modo que su tiempo incluye las transformaciones necesarias y las reglas de calidad. Los Parquet ya contienen los datos preparados. Por eso se comparan rutas completas de consulta y no únicamente el costo de decodificar cada formato.

Parquet permite leer las columnas necesarias y aprovechar filtros; la versión particionada permite además descartar carpetas de otros años y departamentos. No se vacían las cachés del sistema o del almacenamiento entre repeticiones y se mide cada alternativa en bloque.

In [19]:
# Consulta para comparar tiempos de consulta

def consulta(df):
    return (
        df
        .filter(
            (F.col("anio_publicacion") == 2022)
            & (F.col("departamento_entidad") == "Antioquia")
        )
        .groupBy("modalidad_contratacion")
        .agg(
            F.count("*").alias("numero_procesos"),
            F.sum("precio_base").alias("precio_base_total"),
        )
        .orderBy(F.desc("numero_procesos"))
    )

In [36]:
def medir_varias_veces(nombre, df, reps=3):
    tiempos = []

    for i in range(reps):
        inicio = perf_counter()
        consulta(df).collect()
        tiempo = perf_counter() - inicio

        tiempos.append(tiempo)
        print(f"{nombre} — ejecución {i + 1}: {tiempo:.2f} s")

    print(f"{nombre} — mediana: {median(tiempos):.2f} s")
    return tiempos

In [22]:
csv_df = bronze
unpartitioned_df = spark.read.parquet(
    BRONZE_PARQUET_UNPARTITIONED
)
partitioned_df = spark.read.parquet(
    BRONZE_PARQUET
)

In [64]:
# Mediana de tiempo de consulta - CSV

resultado_csv = medir_varias_veces(
    "CSV",
    csv_df,
    reps=3
)

CSV — ejecución 1: 905.59 s
CSV — ejecución 2: 839.71 s
CSV — ejecución 3: 791.84 s
CSV — mediana: 839.71 s


In [38]:
# Mediana de tiempo de consulta - parquet no particionado

resultado_parquet = medir_varias_veces(
    "Parquet sin particionar",
    unpartitioned_df,
    reps=20
)

Parquet sin particionar — ejecución 1: 1.19 s
Parquet sin particionar — ejecución 2: 1.30 s
Parquet sin particionar — ejecución 3: 1.34 s
Parquet sin particionar — ejecución 4: 0.97 s
Parquet sin particionar — ejecución 5: 0.98 s
Parquet sin particionar — ejecución 6: 0.91 s
Parquet sin particionar — ejecución 7: 1.16 s
Parquet sin particionar — ejecución 8: 1.18 s
Parquet sin particionar — ejecución 9: 0.91 s
Parquet sin particionar — ejecución 10: 0.93 s
Parquet sin particionar — ejecución 11: 1.15 s
Parquet sin particionar — ejecución 12: 0.92 s
Parquet sin particionar — ejecución 13: 0.79 s
Parquet sin particionar — ejecución 14: 0.92 s
Parquet sin particionar — ejecución 15: 1.08 s
Parquet sin particionar — ejecución 16: 0.81 s
Parquet sin particionar — ejecución 17: 0.92 s
Parquet sin particionar — ejecución 18: 1.06 s
Parquet sin particionar — ejecución 19: 1.03 s
Parquet sin particionar — ejecución 20: 0.94 s
Parquet sin particionar — mediana: 0.98 s


In [40]:
# Mediana de tiempo de consulta - parquet particionado

resultado_particionado = medir_varias_veces(
    "Parquet particionado",
    partitioned_df,
    reps=20
)

Parquet particionado — ejecución 1: 0.39 s
Parquet particionado — ejecución 2: 0.26 s
Parquet particionado — ejecución 3: 0.24 s
Parquet particionado — ejecución 4: 0.24 s
Parquet particionado — ejecución 5: 0.24 s
Parquet particionado — ejecución 6: 0.22 s
Parquet particionado — ejecución 7: 0.25 s
Parquet particionado — ejecución 8: 0.28 s
Parquet particionado — ejecución 9: 0.27 s
Parquet particionado — ejecución 10: 0.29 s
Parquet particionado — ejecución 11: 0.28 s
Parquet particionado — ejecución 12: 0.26 s
Parquet particionado — ejecución 13: 0.31 s
Parquet particionado — ejecución 14: 0.36 s
Parquet particionado — ejecución 15: 0.32 s
Parquet particionado — ejecución 16: 0.26 s
Parquet particionado — ejecución 17: 0.31 s
Parquet particionado — ejecución 18: 0.29 s
Parquet particionado — ejecución 19: 0.29 s
Parquet particionado — ejecución 20: 0.26 s
Parquet particionado — mediana: 0.27 s


#### Resumen de medianas de consulta

La tabla compara las tres alternativas. Si las mediciones ya se ejecutaron en la sesión, calcula la mediana de sus listas de tiempos; en un kernel nuevo muestra las medianas guardadas en las salidas del archivo recibido, con su precisión original de dos decimales. Así se puede consultar el resumen sin repetir la lectura del CSV.

In [66]:
# Tabla comparativa de las medianas de consulta

resultados_consultas = {
    'CSV (lectura y limpieza)': resultado_csv,
    'Parquet sin particionar': resultado_parquet,
    'Parquet particionado': resultado_particionado,
}

print(f"{'Conjunto de datos':<32} {'Repeticiones':>12} {'Mediana (s)':>14}")
print("-" * 62)
for nombre, resultados in resultados_consultas.items():
    print(f"{nombre:<32} {len(resultados):>12} {median(resultados):>14.2f}")


Conjunto de datos                Repeticiones    Mediana (s)
--------------------------------------------------------------
CSV (lectura y limpieza)                    3         839.71
Parquet sin particionar                    20           0.98
Parquet particionado                       20           0.27


### Comparación de tamaños de los archivos

Se consulta el tamaño total y el número de archivos de cada ruta mediante el sistema de archivos de Hadoop. La reducción porcentual toma el tamaño del CSV como referencia.

Esta comparación incluye tanto el cambio de formato y compresión como la limpieza de datos y las columnas derivadas de bronze; no aísla exclusivamente la compresión. El particionamiento también puede aumentar la cantidad de archivos y su sobrecosto, aunque acelere las consultas que aprovechan esas particiones.

In [32]:
def obtener_tamano(ruta):
    jvm = spark.sparkContext._jvm
    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

    path = jvm.org.apache.hadoop.fs.Path(ruta)
    fs = path.getFileSystem(hadoop_conf)

    resumen = fs.getContentSummary(path)

    bytes_totales = resumen.getLength()
    numero_archivos = resumen.getFileCount()

    return {
        "ruta": ruta,
        "archivos": numero_archivos,
        "bytes": bytes_totales,
        "MB": bytes_totales / (1024 ** 2),
        "GB": bytes_totales / (1024 ** 3),
    }

In [33]:
# Comparación de tamaños de los diferentes archivos

tamanos = [
    ("CSV", obtener_tamano(INPUT_CSV)),
    (
        "Parquet sin particionar",
        obtener_tamano(BRONZE_PARQUET_UNPARTITIONED)
    ),
    (
        "Parquet particionado",
        obtener_tamano(BRONZE_PARQUET)
    ),
]

print(f"{'Formato':<30} {'Archivos':>10} {'MB':>15}")
print("-" * 60)

for nombre, datos in tamanos:
    print(
        f"{nombre:<30} "
        f"{datos['archivos']:>10} "
        f"{datos['MB']:>15.2f} "
    )

Formato                          Archivos              MB
------------------------------------------------------------
CSV                                     1         9390.17 
Parquet sin particionar                75         1638.45 
Parquet particionado                  396         1398.49 


In [34]:
# Comparación de reducción

csv_mb = tamanos[0][1]["MB"]
parquet_mb = tamanos[1][1]["MB"]
particionado_mb = tamanos[2][1]["MB"]

reduccion_parquet = (1 - parquet_mb / csv_mb) * 100
reduccion_particionado = (1 - particionado_mb / csv_mb) * 100

print(
    f"Reducción CSV → Parquet sin particionar: "
    f"{reduccion_parquet:.2f}%"
)

print(
    f"Reducción CSV → Parquet particionado: "
    f"{reduccion_particionado:.2f}%"
)

Reducción CSV → Parquet sin particionar: 82.55%
Reducción CSV → Parquet particionado: 85.11%


### Comparación de tiempos de escritura

Se utiliza un subconjunto de hasta **10.000 filas**, almacenado en caché y materializado antes de medir. Así se reduce la influencia de la lectura de origen sobre las escrituras.

Cada formato se escribe cinco veces en rutas de benchmark, cambiando aleatoriamente el orden en cada repetición. La escritura particionada incluye redistribución por año y departamento, además de la creación de archivos.

In [41]:
# Configuraciones iniciales para comparaciones de tiempos de escritura

NUM_FILAS = 10_000
OUTPUT_PARTITIONS_BENCHMARK = 4
REPETICIONES = 5
RUTA_BENCHMARK = "s3a://rawdata/benchmark_escritura"

base = spark.read.parquet(BRONZE_PARQUET_UNPARTITIONED)

subconjunto = (
    base
    .limit(NUM_FILAS)
    .cache()
)

# Materializa el subconjunto antes de medir
filas_reales = subconjunto.count()

print(f"Filas del subconjunto: {filas_reales:,}")

Filas del subconjunto: 10,000


In [42]:
# Definición de funciones para la escritura

def escribir_csv(df, ruta):
    (
        df
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(ruta)
    )


def escribir_parquet(df, ruta):
    (
        df
        .write
        .mode("overwrite")
        .option("compression", "snappy")
        .parquet(ruta)
    )


def escribir_parquet_particionado(df, ruta):
    (
        df
        .repartition(
            OUTPUT_PARTITIONS_BENCHMARK,
            "anio_publicacion",
            "departamento_entidad"
        )
        .write
        .mode("overwrite")
        .option("compression", "snappy")
        .partitionBy(
            "anio_publicacion",
            "departamento_entidad"
        )
        .parquet(ruta)
    )

In [109]:
# Cálculo de tiempos de escritura por cada tipo de archivo

escritores = {
    "CSV": escribir_csv,
    "Parquet sin particionar": escribir_parquet,
    "Parquet particionado": escribir_parquet_particionado,
}

tiempos = {
    nombre: []
    for nombre in escritores
}

for repeticion in range(1, REPETICIONES + 1):
    formatos = list(escritores.items())

    # Cambia el orden en cada repetición
    random.shuffle(formatos)

    for nombre, escritor in formatos:
        nombre_ruta = (
            nombre.lower()
            .replace(" ", "_")
            .replace("ó", "o")
        )

        ruta = (
            f"{RUTA_BENCHMARK}/"
            f"{nombre_ruta}/repeticion_{repeticion}"
        )

        inicio = perf_counter()

        escritor(subconjunto, ruta)

        segundos = perf_counter() - inicio
        tiempos[nombre].append(segundos)

        print(
            f"{nombre} — repetición {repeticion}: "
            f"{segundos:.2f} s "
            f"({segundos / 60:.2f} min)"
        )

Parquet sin particionar — repetición 1: 0.66 s (0.01 min)
CSV — repetición 1: 0.55 s (0.01 min)
Parquet particionado — repetición 1: 1.96 s (0.03 min)
Parquet particionado — repetición 2: 1.81 s (0.03 min)
Parquet sin particionar — repetición 2: 0.59 s (0.01 min)
CSV — repetición 2: 0.51 s (0.01 min)
CSV — repetición 3: 0.57 s (0.01 min)
Parquet particionado — repetición 3: 1.86 s (0.03 min)
Parquet sin particionar — repetición 3: 0.56 s (0.01 min)
Parquet sin particionar — repetición 4: 0.50 s (0.01 min)
Parquet particionado — repetición 4: 2.27 s (0.04 min)
CSV — repetición 4: 0.69 s (0.01 min)
CSV — repetición 5: 0.63 s (0.01 min)
Parquet particionado — repetición 5: 1.95 s (0.03 min)
Parquet sin particionar — repetición 5: 0.87 s (0.01 min)


In [110]:
# Comparación de medianas de tiempo de lectura

print(
    f"{'Formato':<30}"
    f"{'Tiempos (s)':<35}"
    f"{'Mediana (s)':>15}"
)

print("-" * 80)

for nombre, valores in tiempos.items():
    valores_texto = ", ".join(f"{v:.2f}" for v in valores)

    print(
        f"{nombre:<30}"
        f"{valores_texto:<35}"
        f"{median(valores):>15.2f}"
    )

Formato                       Tiempos (s)                            Mediana (s)
--------------------------------------------------------------------------------
CSV                           0.55, 0.51, 0.57, 0.69, 0.63                  0.57
Parquet sin particionar       0.66, 0.59, 0.56, 0.50, 0.87                  0.59
Parquet particionado          1.96, 1.81, 1.86, 2.27, 1.95                  1.95


### Consultas analíticas

Como aplicación de los datos preparados, se generan dos resultados agregados: procesos y precios base por año y departamento, y estadísticas de precio base por modalidad de contratación. Se guardan en Parquet en la zona `procesado`, para poder consultarlos sin repetir la agregación.

Estos resultados usan `precio_base`, no el valor efectivamente adjudicado. La suma y el promedio omiten precios nulos, mientras que `count("*")` cuenta todos los registros que superan los filtros de cada consulta.

In [111]:
# Consulta analítica de departamentos y año para escritura

RUTA_CONSULTA_DEPARTAMENTO = (
    "s3a://rawdata/procesado/precio_base_departamentos_anios"
)

consulta_departamento = (
    bronze
    .filter(
        F.col("anio_publicacion").isNotNull()
        & F.col("departamento_entidad").isNotNull()
    )
    .groupBy(
        "anio_publicacion",
        "departamento_entidad",
    )
    .agg(
        F.count("*").alias("cantidad_procesos"),
        F.sum("precio_base").alias("precio_base_total"),
        F.avg("precio_base").alias("precio_base_promedio"),
    )
)

(
    consulta_departamento
    .write
    .mode("overwrite")
    .option("compression", "snappy")
    .parquet(RUTA_CONSULTA_DEPARTAMENTO)
)

In [113]:
# Consulta analítica modalidad de contratación y precio_base para escritura

RUTA_CONSULTA_MODALIDAD = (
    "s3a://rawdata/procesado/precio_base_modalidad"
)

consulta_modalidad = (
    bronze
    .filter(F.col("modalidad_contratacion").isNotNull())
    .groupBy("modalidad_contratacion")
    .agg(
        F.count("*").alias("cantidad_procesos"),
        F.sum("precio_base").alias("precio_base_total"),
        F.avg("precio_base").alias("precio_base_promedio"),
        F.min("precio_base").alias("precio_base_minimo"),
        F.max("precio_base").alias("precio_base_maximo"),
    )
    .orderBy(F.desc("cantidad_procesos"))
)

(
    consulta_modalidad
    .write
    .mode("overwrite")
    .option("compression", "snappy")
    .parquet(RUTA_CONSULTA_MODALIDAD)
)

### Conclusiones

Las medianas registradas fueron **839,71 s para CSV**, **0,98 s para Parquet sin particionar** y **0,27 s para Parquet particionado**. Para esta consulta sobre Antioquia en 2022, el Parquet particionado presentó el menor tiempo; el resultado es consistente con la posibilidad de descartar las carpetas de otros años y departamentos.

La diferencia frente al CSV refleja también el costo de ejecutar su limpieza y las reglas de calidad durante la consulta, mientras que los Parquet ya están preparados. Por eso no debe atribuirse toda la mejora al formato. Además, se hicieron tres repeticiones para CSV y veinte para cada Parquet, sin vaciar las cachés del sistema entre ejecuciones.

La organización del almacenamiento debe evaluarse junto con el espacio ocupado y el costo de escritura. Particionar puede requerir más trabajo al guardar y ahorrar trabajo en consultas posteriores que filtren por esas columnas.